# Pedro Benchmark Report Pipeline

This notebook runs the repository benchmark and reporting workflow from Python cells. It mirrors the local command-line flow in `docs/local_benchmark_quickstart.md`: install dependencies, run a small benchmark, generate per-run charts/PDFs, run efficiency measurements, and optionally build the project-level snapshot report.

Start with the smoke settings first. Increase limits only after the model loads and the datasets are accessible.

## 1. Setup

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

def find_project_root(start=None):
    start = Path(start or os.getcwd()).resolve()
    for path in [start, *start.parents]:
        if (path / "scripts" / "run_benchmarks.py").exists():
            return path
    raise RuntimeError("Could not find repository root. Run this notebook inside the cloned repo.")

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# Install dependencies when running in a fresh Colab or local virtual environment.
# This can take several minutes.
INSTALL_DEPENDENCIES = False

if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)

In [ ]:
import platform
import shutil

try:
    import torch
except Exception as exc:
    torch = None
    print(f"PyTorch is not available yet: {exc}")

print(f"Python: {platform.python_version()}")
print(f"Free disk: {shutil.disk_usage(PROJECT_ROOT).free / (1024**3):.1f} GB")

if torch is not None:
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"MPS available: {torch.backends.mps.is_available() if hasattr(torch.backends, 'mps') else False}")

## 2. Hugging Face Access

LLaMA models and GPQA may require Hugging Face access approval. Run the login cell only when needed.

In [ ]:
# Optional login for gated models/datasets.
# from huggingface_hub import login
# from getpass import getpass
# login(token=getpass("Hugging Face token: "))

## 3. Choose Run Settings

In [ ]:
# Use a local model path if you already downloaded weights into models/.
# Otherwise set MODEL_ID to a Hugging Face model id such as "meta-llama/Llama-3.2-1B-Instruct".
MODEL_ID = "./models/llama-3.2-1b"
DEVICE = "auto"       # auto, cuda, cpu, or mps
PRECISION = "fp16"    # fp16, bf16, int4, or int2

# Smoke-test settings. Increase LIMIT for a final run.
BENCHMARKS = ["hellaswag", "arc"]
LIMIT = 5

print("Model exists locally:", Path(MODEL_ID).exists() if MODEL_ID.startswith("./") else "remote HF id")

## 4. Run a Benchmark and Generate Per-Run Artifacts

This calls `scripts/run_benchmarks.py`. By default the script writes raw JSON plus accuracy/runtime PNG charts and a PDF report into `results/`.

In [ ]:
benchmark_cmd = [
    sys.executable,
    "scripts/run_benchmarks.py",
    "--model-id", MODEL_ID,
    "--benchmarks", *BENCHMARKS,
    "--limit", str(LIMIT),
    "--precision", PRECISION,
    "--device", DEVICE,
]

print(" ".join(benchmark_cmd))
subprocess.run(benchmark_cmd, check=True)

In [ ]:
import json
from IPython.display import Image, display

raw_results = sorted((PROJECT_ROOT / "results" / "raw").glob("benchmark_*.json"), key=lambda p: p.stat().st_mtime)
latest_benchmark = raw_results[-1]
print(f"Latest benchmark JSON: {latest_benchmark}")

with latest_benchmark.open("r", encoding="utf-8") as f:
    benchmark_data = json.load(f)

for item in benchmark_data["benchmarks"]:
    print(f"{item['name']}: accuracy={item['accuracy']:.3f} ({item['correct']}/{item['num_examples']}), duration={item['duration_sec']:.2f}s")

run_id = benchmark_data["run_id"]
report_dir = PROJECT_ROOT / "results" / "reports"
for image_path in [report_dir / f"{run_id}_accuracy.png", report_dir / f"{run_id}_runtime.png"]:
    if image_path.exists():
        display(Image(filename=str(image_path)))

pdf_path = report_dir / f"{run_id}_report.pdf"
print(f"Per-run PDF: {pdf_path if pdf_path.exists() else 'not found'}")

## 5. Run Efficiency Measurements

Efficiency runs record TTFT, latency per token, throughput, and memory metadata. The project snapshot report expects each target model to have a benchmark run and an efficiency run.

In [ ]:
RUN_EFFICIENCY = False

if RUN_EFFICIENCY:
    efficiency_cmd = [
        sys.executable,
        "scripts/run_efficiency.py",
        "--model-id", MODEL_ID,
        "--precision", PRECISION,
        "--device", DEVICE,
        "--num-prompts", "2",
        "--warmup-runs", "0",
        "--timed-runs", "1",
        "--max-new-tokens", "32",
    ]
    print(" ".join(efficiency_cmd))
    subprocess.run(efficiency_cmd, check=True)
else:
    print("Set RUN_EFFICIENCY = True to run efficiency measurements.")

In [ ]:
efficiency_results = sorted((PROJECT_ROOT / "results" / "raw").glob("efficiency_*.json"), key=lambda p: p.stat().st_mtime)

if efficiency_results:
    latest_efficiency = efficiency_results[-1]
    print(f"Latest efficiency JSON: {latest_efficiency}")
    with latest_efficiency.open("r", encoding="utf-8") as f:
        efficiency_data = json.load(f)
    print(json.dumps(efficiency_data["metrics"], indent=2))
else:
    print("No efficiency JSON files found yet.")

## 6. Optional Project Snapshot Report

`scripts/generate_report.py` builds a multi-model PDF from the latest matching benchmark and efficiency runs for LLaMA 3.2 1B, LLaMA 3.2 3B, and Phi-3-mini. It requires all expected raw JSON files to exist first.

In [ ]:
GENERATE_PROJECT_REPORT = False

if GENERATE_PROJECT_REPORT:
    subprocess.run([sys.executable, "scripts/generate_report.py"], check=True)
else:
    print("Set GENERATE_PROJECT_REPORT = True after you have benchmark and efficiency runs for all target models.")

In [ ]:
snapshot_report = PROJECT_ROOT / "results" / "reports" / "project_snapshot_report.pdf"
print(f"Snapshot report: {snapshot_report if snapshot_report.exists() else 'not generated yet'}")

## 7. Full Baseline Commands

Use these settings when the smoke run works and you want comparable report inputs:

- Benchmarks: `hellaswag arc gpqa gsm8k`
- Benchmark precision: `fp16`
- Efficiency prompts: `5`
- Efficiency timed runs: `3`
- Efficiency max new tokens: `64`

Run the benchmark and efficiency cells once per model: `./models/llama-3.2-1b`, `./models/llama-3.2-3b`, and `./models/phi-3-mini`.